In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/GAID_MASTER_V2_COMPILATION_FINAL.csv")

In [2]:
df.columns = df.columns.str.strip()

In [3]:
df["Year"] = df["Year"].astype(int)
df["Country"] = df["Country"].astype(str).str.strip()
df["ISO3"] = df["ISO3"].astype(str).str.strip()
df["Metric"] = df["Metric"].astype(str).str.strip()
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

In [4]:
df_clean = df.dropna(subset=["Value"]).copy()

In [5]:
df_clean = df_clean.drop_duplicates()

In [6]:
df_clean.shape
df_clean.isna().sum()
df_clean.duplicated().sum()

np.int64(0)

In [ ]:
#Creación del Modelo Estrella
#Dimensión: Dim_Country
dim_country = (
    df_clean[["Country", "ISO3"]]
    .drop_duplicates()
    .sort_values("Country")
    .reset_index(drop=True)
)

dim_country["CountryKey"] = dim_country.index + 1

dim_country = dim_country[["CountryKey", "Country", "ISO3"]]

In [9]:
#Dimension: Dim_Year
dim_year = (
    df_clean[["Year"]]
    .drop_duplicates()
    .sort_values("Year")
    .reset_index(drop=True)
)

dim_year["YearKey"] = dim_year["Year"]
dim_year = dim_year[["YearKey", "Year"]]

In [10]:
#Dimension: Dim_Metric
dim_metric = (
    df_clean[["Metric", "Source_Category"]]
    .drop_duplicates()
    .sort_values("Metric")
    .reset_index(drop=True)
)

dim_metric["MetricKey"] = dim_metric.index + 1

dim_metric = dim_metric[["MetricKey", "Metric", "Source_Category"]]

In [11]:
#Dimension: Dim_Source
dim_source = (
    df_clean[["Source", "Dataset", "Source_File", "Source_Type", "Source_Year"]]
    .drop_duplicates()
    .sort_values("Source")
    .reset_index(drop=True)
)

dim_source["SourceKey"] = dim_source.index + 1

dim_source = dim_source[
    ["SourceKey", "Source", "Dataset", "Source_File", "Source_Type", "Source_Year"]
]

In [12]:
#Creacion de Fact Table
fact = df_clean.merge(dim_country, on=["Country", "ISO3"], how="left")
fact = fact.merge(dim_year, on="Year", how="left")
fact = fact.merge(dim_metric, on=["Metric", "Source_Category"], how="left")
fact = fact.merge(
    dim_source,
    on=["Source", "Dataset", "Source_File", "Source_Type", "Source_Year"],
    how="left"
)

In [13]:
#Selección de columnas finales
fact_ai_indicators = fact[
    [
        "YearKey",
        "CountryKey",
        "MetricKey",
        "SourceKey",
        "Value"
    ]
].copy()

In [7]:
#Validación de que no haya claves nulas
fact_ai_indicators.isna().sum()

NameError: name 'fact_ai_indicators' is not defined

In [15]:
#Exportar a CSV
dim_country.to_csv("../data/processed/dim_country.csv", index=False)
dim_year.to_csv("../data/processed/dim_year.csv", index=False)
dim_metric.to_csv("../data/processed/dim_metric.csv", index=False)
dim_source.to_csv("../data/processed/dim_source.csv", index=False)
fact_ai_indicators.to_csv("../data/processed/fact_ai_indicators.csv", index=False)

In [8]:
df_clean.to_csv("../data/processed/gaid_clean.csv", index=False)